# 01 - Construcción del dataset etiquetado

Genera el dataset de pares *(posición, evaluación)* que alimenta el
entrenamiento: descarga partidas de Lichess, las filtra, muestrea posiciones y
las etiqueta con Stockfish a profundidad fija.

Cubre las tareas 3.2 a 3.6 del WBS y cierra los requerimientos 1.1, 1.2, 1.3 y 2.3.

## Cómo funciona

El pipeline hace **dos pasadas** por cada dump:

1. **Extracción** — recorre el dump comprimido (decenas de GB) una sola vez por
   streaming y guarda en un PGN chico solo las partidas que pasan el filtro. Ese
   extracto se publica en el Hub, así que esta pasada ocurre **una vez en la vida
   del proyecto**: cualquier máquina posterior lo baja en minutos en lugar de
   rehacer la hora de streaming.
2. **Etiquetado** — lee ese extracto, muestrea 4 posiciones por partida (2 con
   blancas al turno y 2 con negras), las evalúa con Stockfish y escribe shards
   Parquet, subiendo cada uno apenas se cierra.

## Dónde vive cada cosa

Todo lo durable está en Hugging Face. El disco local es solo un cache
descartable, y **no se usa Google Drive**: no hay unidad que montar, y la
corrida se puede continuar desde cualquier máquina.

| Qué | Dónde |
|---|---|
| Shards etiquetados | `ceia-chess-eval` — el entregable |
| Extracto filtrado | `ceia-chess-work` |
| Estado de reanudación | `ceia-chess-work` |
| Cache de trabajo | `/content` — se pierde y no importa |

La deduplicación no se guarda en ningún archivo: se reconstruye leyendo la
columna `pos_key` de los shards ya publicados. El dataset es su propio registro
de lo que contiene, así que no hay un segundo archivo que se desincronice.

> **Runtime: CPU, no GPU.** Stockfish es puro CPU y los runtimes con GPU de
> Colab traen *menos* vCPUs: elegir GPU acá es más lento y además gasta cuota
> que conviene reservar para el entrenamiento.

## 1. Entorno

In [ ]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess, importlib
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# pip registra el paquete editable con un .pth, y los .pth solo se procesan al
# arrancar el interprete: un kernel que ya esta corriendo no lo ve. Agregar src/
# a sys.path lo hace visible sin tener que reiniciar el runtime.
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# Stockfish con version fija (queda registrada en cada fila del dataset).
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())

In [ ]:
from chessdl.colab import describe_runtime

runtime = describe_runtime("stockfish")
print(runtime.summary())

## 2. Verificación del código (tests)

Antes de gastar horas de etiquetado conviene comprobar que el código hace lo que
dice. `pytest` descubre solo los archivos `tests/test_*.py`; no hay que importar
ni llamar nada a mano.

La salida de esta celda es la evidencia de los requerimientos de testing (3.1 y
3.2) para la memoria.

In [ ]:
!{sys.executable} -m pytest -q

Para ver el nombre de cada test, o correr un subconjunto:

```
!{sys.executable} -m pytest tests/test_encoding.py -v     # un archivo, test por test
!{sys.executable} -m pytest -k mirror -v                  # los 3 tests del espejado
!{sys.executable} -m pytest --collect-only -q             # listar sin ejecutar
```

## 3. Credenciales de Hugging Face

El token se lee del panel de **Secrets** de Colab (icono de la llave, a la
izquierda): crear un secreto `HF_TOKEN` con permiso de **escritura** y habilitarlo
para este notebook. Nunca pegar el token en una celda.

Como ahora el Hub guarda también el extracto y el estado, sin token la corrida
no puede reanudarse en otra máquina.

In [ ]:
from huggingface_hub import HfApi
from chessdl import hf
from chessdl.config import load_config

cfg = load_config()
token = hf.get_token()

if token is None:
    print("No hay token. Revisa el panel de Secrets y que el toggle de acceso")
    print("para este notebook este activado.")
else:
    info = HfApi(token=token).whoami()
    permiso = (info.get("auth", {}).get("accessToken", {}) or {}).get("role", "?")
    print("Usuario          :", info.get("name"))
    print("Permiso          :", permiso, "  <-- tiene que decir 'write'")
    print("Dataset          :", cfg.output.hf_repo_id)
    print("Datos de trabajo :", cfg.output.hf_work_repo_id)
    if info.get("name") != cfg.output.hf_namespace:
        print()
        print("OJO: el usuario del token no coincide con el namespace configurado.")

## 4. Parámetros

Se imprimen para que queden registrados en la salida del notebook: es la
evidencia de con qué configuración se generó cada versión del dataset
(requerimiento 2.3).

In [ ]:
from chessdl.data.labeling import engine_version

print("Dumps               :", cfg.source.dumps)
print("ELO minimo          :", cfg.filter.min_elo)
print("Controles de tiempo :", cfg.filter.time_controls)
print("Plies minimos       :", cfg.filter.min_plies)
print("Posiciones/partida  :", cfg.sampling.positions_per_game, "(balanceadas por turno)")
print("Semilla de muestreo :", cfg.sampling.seed)
print("Profundidad SF      :", cfg.labeling.depth)
print("Workers             :", cfg.labeling.resolved_workers())
print("Normalizacion       : value = tanh(cp /", cfg.normalization.scale,
      "), recorte +-", cfg.normalization.cp_clip)
print("Partidas por shard  :", cfg.output.games_per_shard)
print()
sf_version = engine_version(cfg.labeling)
print("Motor:", sf_version)

### Cuántos workers de verdad

Colab suele reportar más CPUs de las que asigna. Si `resolved_workers()` supera
las CPUs reales, se lanzan más Stockfish de los que entran y compiten entre sí.

In [ ]:
def cuota_cgroup():
    v2 = Path("/sys/fs/cgroup/cpu.max")
    if v2.exists():
        cuota, periodo = v2.read_text().split()
        if cuota != "max":
            return int(cuota) / int(periodo)
    v1q, v1p = Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us"), Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us")
    if v1q.exists() and v1p.exists():
        q = int(v1q.read_text())
        if q > 0:
            return q / int(v1p.read_text())
    return None

print("os.cpu_count()          :", os.cpu_count())
print("CPUs realmente asignadas:", len(os.sched_getaffinity(0)))
print("Cuota del cgroup        :", cuota_cgroup() or "sin limite")
print("Workers del pipeline    :", cfg.labeling.resolved_workers())

## 5. Primera tanda

`run_build` sincroniza los shards ya publicados, reconstruye la deduplicación a
partir de ellos, trae el estado del Hub y sigue desde donde quedó.

La **primera** ejecución incluye la extracción del dump: alrededor de una hora
de streaming, sin barra de Stockfish todavía. Se publica al terminar, así que
las corridas siguientes —en esta máquina o en cualquier otra— la saltean.

`max_shards` acota cuánto se hace por sesión.

In [ ]:
import time
from chessdl.data import pipeline

inicio = time.time()
resumen = pipeline.run_build(
    cfg,
    sf_version=sf_version,
    max_shards=5,
    progress=True,
)
transcurrido = time.time() - inicio

print()
print(resumen.summary())
if resumen.n_positions:
    print(f"\nVelocidad: {resumen.n_positions / transcurrido:.1f} posiciones/segundo")

### Rendimiento por shard

El techo teórico son `partidas_por_shard × posiciones_por_partida`. Lo que falta
se reparte entre partidas de menos de 20 plies y posiciones duplicadas.

In [ ]:
if resumen.shards:
    print(f"{'shard':>6}{'partidas':>10}{'posiciones':>12}{'duplicadas':>12}{'descartadas':>13}")
    print("-" * 53)
    for s in resumen.shards:
        print(f"{s.index:>6}{s.n_games:>10}{s.n_positions:>12}{s.n_duplicates:>12}{s.n_dropped:>13}")

    techo = len(resumen.shards) * cfg.output.games_per_shard * cfg.sampling.positions_per_game
    print(f"\nRendimiento: {resumen.n_positions / techo:.1%} del techo teorico")
else:
    print("Ningun shard en esta tanda.")

## 6. Seguir hasta el objetivo

Este bucle repite tandas hasta llegar al objetivo o agotar el extracto. Es
**reanudable**: si Colab se desconecta, se reconecta, se corren las celdas 1 a 4
y se relanza esta misma celda.

A ~19 posiciones/segundo, 500.000 posiciones son unas 7,5 horas. Ese número —y
no las 2.000.000 que puse originalmente— es un objetivo razonable para la
primera campaña; el plan no fija ningún tamaño, y acotarlo es justamente la
mitigación prevista para el Riesgo 1.

In [ ]:
from chessdl.data.state import PipelineState

OBJETIVO = 500_000

while True:
    resumen = pipeline.run_build(cfg, sf_version=sf_version, max_shards=5, progress=True)
    estado = PipelineState.load(cfg.output.state_path,
                                repo_id=cfg.output.hf_work_repo_id, token=token)
    print(f"\n>>> acumulado: {estado.total_positions:,} posiciones\n")

    if not resumen.shards:
        print("Extracto agotado: no quedan partidas en este dump.")
        break
    if estado.total_positions >= OBJETIVO:
        print("Objetivo alcanzado.")
        break

## 7. Validación de integridad

Los chequeos del requerimiento 3.2 sobre lo generado hasta ahora: sin posiciones
duplicadas, balance de color, rangos válidos, sin nulos, ELO por encima del
umbral, FENs legales y no terminales, y coherencia de signo entre los dos puntos
de vista.

In [ ]:
from chessdl.data import schema
from chessdl.data.validate import describe_table, validate_table

tabla = schema.read_dataset(schema.shard_paths(cfg.output.local_dir))
reporte = validate_table(tabla, cfg)
print(reporte.summary())

In [ ]:
stats = describe_table(tabla)
for clave, valor in stats.items():
    print(f"{clave:>22}: {valor:,.4f}" if isinstance(valor, float) else f"{clave:>22}: {valor:,}")

## 8. Progreso acumulado

In [ ]:
estado = PipelineState.load(cfg.output.state_path,
                            repo_id=cfg.output.hf_work_repo_id, token=token)
for nombre, dump_state in estado.dumps.items():
    print(f"{nombre}: {dump_state.shards_done} shards, "
          f"{dump_state.positions_written:,} posiciones "
          f"({dump_state.games_accepted:,} partidas aceptadas de {dump_state.games_seen:,})")
print()
print(f"Total acumulado: {estado.total_positions:,} posiciones")
print(f"Dataset        : https://huggingface.co/datasets/{cfg.output.hf_repo_id}")

**Próximo paso:** `02_dataset_eda.ipynb` para el análisis exploratorio (WBS 3.5).